# Gold Layer ETL (PySpark)

Automated PySpark pipeline building the **Gold Layer** for Kolumbus/Entur bus analytics directly from cleaned Silver datasets.

### Source Tables (Silver & v6 Cleaned Data):
1. `clean_stop_visits` (87,279 rows)
2. `expected_per_stop` (137,806 rows)
3. `per_stop_evidence` (21,737 rows)
4. `stop_event_status` (87,092 rows)
5. `stop_gps_status` (86,241 rows)
6. `time_lost_null` (47,480 rows)
7. `trip_vehicle` (26,006 rows)

### Output Tables (Gold Layer):
- **Dimensions**: `dim_line`, `dim_stop`, `dim_date`, `dim_hour_band`, `gold_buckets`
- **Aggregations & Cubes**: `gold_visits_cube`, `gold_stop_cube`, `gold_stop_order_delay`, `fact_stop_visits`
- **Operations Analytics**: `gold_segment_runtime`, `gold_headway`
- **Evidence & Fleet**: `gold_fleet_activity`, `gold_stop_evidence`, `gold_time_lost_reference`
- **Metadata & Audit**: `gold_data_quality`, `gold_meta`
- **Storage**: Parquet files + integrated `gold.duckdb` file

In [5]:
import json
import os
from pathlib import Path
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F

# ---------------- Directory Settings ----------------
SILVER_DIR = os.environ.get("SILVER_DIR", ".")       # Folder containing cleaned silver parquet files
GOLD_DIR = os.environ.get("GOLD_DIR", "gold")            # Output gold folder

# ---------------- Analysis Configuration ----------------
LOCAL_TZ = "Europe/Oslo"
EARLY_S, LATE_S = -60, 300                      # On-time threshold (seconds): -60s <= delay < 300s
DELAY_EDGES = [-120, -60, 0, 60, 180, 300, 600]  # Delay histogram buckets
TL_EDGES = [0, 5, 10, 20, 40]                   # Time lost histogram buckets
HOUR_BANDS = {"night": (0, 5), "am_peak": (6, 8), "midday": (9, 14), "pm_peak": (15, 17), "evening": (18, 23)}
SEG_MAX_S = 1800                                # Max segment runtime (30 mins)
HEADWAY_MIN_S, HEADWAY_MAX_S = 120, 3600        # Valid scheduled headway range (2m to 60m)
BUNCH_RATIO, GAP_RATIO = 0.5, 1.5               # Bunching < 0.5x, Gap > 1.5x scheduled headway

BAND_OF_HOUR = {h: b for b, (lo, hi) in HOUR_BANDS.items() for h in range(lo, hi + 1)}
OUT = Path(GOLD_DIR)
OUT.mkdir(parents=True, exist_ok=True)
print("Configuration loaded. Output directory:", OUT.resolve())

Configuration loaded. Output directory: D:\coding\cloud_computing\apis\Entur-Buspulse\tables\parquet_2026-10-24\clean\gold


In [2]:
spark = (SparkSession.builder
         .appName("buspulse-gold-pipeline")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.legacy.parquet.nanosAsLong", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark Session initialized successfully. Version:", spark.version)

c:\Users\Hameed Computers\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Session initialized successfully. Version: 4.2.0


In [3]:
CHECKS = []      # Reconciliation audit log
TABLES = {}      # Gold table row counts

def check(label, ok, detail=""):
    CHECKS.append({"check": label, "ok": bool(ok), "detail": str(detail)})
    status = "PASS" if ok else "FAIL"
    print(f"{status:4s} | {label} | {detail}")

def read(folder, name, required=True):
    p = Path(folder) / f"{name}.parquet"
    if not p.exists():
        if required:
            raise FileNotFoundError(f"Missing required Parquet file: {p}")
        print(f"Note: {p} not found, skipping dependent transformations.")
        return None
    return spark.read.parquet(str(p))

def dtype(df, name):
    return dict(df.dtypes).get(name)

def to_bool(df, name):
    if name not in df.columns:
        return F.lit(False)
    if dtype(df, name) == "boolean":
        return F.coalesce(F.col(name), F.lit(False))
    s = F.lower(F.trim(F.col(name).cast("string")))
    return F.coalesce(s.isin("true", "1", "t", "yes"), F.lit(False))

def to_utc(df, name):
    if name not in df.columns:
        return F.lit(None).cast("timestamp")
    if dtype(df, name) in ("timestamp", "timestamp_ntz", "date"):
        return F.col(name).cast("timestamp")
    if dtype(df, name) == "bigint":
        return F.timestamp_micros(F.floor(F.col(name) / 1000).cast("long"))
    return F.expr(f"try_cast(`{name}` AS timestamp)")

def num(df, name):
    if name not in df.columns:
        return F.lit(None).cast("double")
    c = F.col(name).cast("double")
    return F.when(F.isnan(c), None).otherwise(c)

def label(df, name):
    if name not in df.columns:
        return F.lit("unknown")
    c = F.col(name).cast("string")
    return F.when(c.isNull() | c.isin("None", "nan", ""), "unknown").otherwise(c)

def local(col_name):
    return F.from_utc_timestamp(F.col(col_name), LOCAL_TZ)

def weekday_mon0(ts):
    return (F.dayofweek(ts) + 5) % 7

def hour_band(hour_col):
    band = F.lit("unknown")
    for b, (lo, hi) in reversed(list(HOUR_BANDS.items())):
        band = F.when(hour_col.between(lo, hi), b).otherwise(band)
    return band

def hist_cols(prefix, column, cond, edges):
    parts = []
    for i in range(len(edges) + 1):
        if i == 0:
            c = f"{column} < {edges[0]}"
        elif i == len(edges):
            c = f"{column} >= {edges[-1]}"
        else:
            c = f"{column} >= {edges[i - 1]} AND {column} < {edges[i]}"
        parts.append(f"SUM(CAST(({cond} AND {c}) AS INT)) AS {prefix}{i}")
    return ",\n  ".join(parts)

def bucket_table(metric, prefix, edges):
    rows = []
    for i in range(len(edges) + 1):
        lo = None if i == 0 else edges[i - 1]
        hi = None if i == len(edges) else edges[i]
        lbl = f"< {hi}" if lo is None else (f">= {lo}" if hi is None else f"{lo} to {hi}")
        rows.append((metric, f"{prefix}{i}", lo, hi, lbl))
    return rows

def emit(name, df, one_file=True):
    for c, t in df.dtypes:
        if t == "timestamp":
            df = df.withColumn(c, F.col(c).cast("timestamp_ntz"))
    path = str(OUT / f"{name}.parquet")
    (df.coalesce(1) if one_file else df).write.mode("overwrite").parquet(path)
    t = spark.read.parquet(path)
    t.createOrReplaceTempView(name)
    TABLES[name] = t.count()
    print(f"  [EMIT] {name}: {TABLES[name]:,} rows written")
    return t

print("Helper functions configured.")

Helper functions configured.


In [6]:
print("Staging clean_stop_visits...")
sv_raw = read(SILVER_DIR, "clean_stop_visits")

# Normalize timestamps and attributes using exact Silver schema column names
sv = sv_raw.withColumns({
    "aimed_arrival": to_utc(sv_raw, "aimed_arrival"),
    "actual_arrival": to_utc(sv_raw, "actual_arrival"),
    "aimed_departure": to_utc(sv_raw, "aimed_departure"),
    "actual_departure": to_utc(sv_raw, "actual_departure"),
    "visit_time": to_utc(sv_raw, "visit_time"),
    "aimed_time": to_utc(sv_raw, "aimed_time"),
    "is_actual": to_bool(sv_raw, "is_actual"),
    "delay_outlier": to_bool(sv_raw, "delay_outlier"),
    "on_time": to_bool(sv_raw, "on_time"),
    "delay_s": num(sv_raw, "delay_s"),
    "delay_arrival_s": num(sv_raw, "delay_arrival_s"),
    "delay_departure_s": num(sv_raw, "delay_departure_s"),
    "line_ref": label(sv_raw, "line_ref"),
    "route_short_name": label(sv_raw, "route_short_name"),
    "route_type_name": label(sv_raw, "route_type_name"),
    "direction_ref": label(sv_raw, "direction_ref"),
    "journey_ref": label(sv_raw, "journey_ref"),
    "stop_ref": label(sv_raw, "stop_ref"),
    "stop_name": label(sv_raw, "stop_name")
})

loc = local("visit_time")
sv = sv.withColumns({
    "local_hour": F.coalesce(F.hour(loc), F.lit(-1)),
    "weekday": F.coalesce(weekday_mon0(loc), F.lit(-1)),
    "service_date": F.col("service_date").cast("string")
})
sv = sv.withColumns({
    "hour_band": hour_band(F.col("local_hour")),
    "valid": F.col("is_actual") & ~F.col("delay_outlier") & F.col("delay_s").isNotNull(),
}).cache()
sv.createOrReplaceTempView("sv")

n_sv = sv.count()
dates = sorted(r[0] for r in sv.select("service_date").distinct().collect() if r[0] is not None)
print(f"Staged clean_stop_visits: {n_sv:,} rows, Service Dates: {dates}")

Staging clean_stop_visits...
Staged clean_stop_visits: 87,279 rows, Service Dates: ['2026-10-24', '2026-10-25', '2026-10-26', '2026-10-27']


In [7]:
# Sanity check on delay_s consistency using actual departure vs aimed departure
d_calc = F.col("actual_departure").cast("double") - F.col("aimed_departure").cast("double")
m = (F.col("is_actual") & F.col("actual_departure").isNotNull() & F.col("aimed_departure").isNotNull()
     & F.col("delay_s").isNotNull())
r = sv.where(m).agg(
    F.avg((F.abs(d_calc - F.col("delay_s")) <= 2).cast("int")).alias("agree"),
    F.avg((F.abs(d_calc + F.col("delay_s")) <= 2).cast("int")).alias("flipped")).first()

agree = r["agree"] if (r and r["agree"] is not None) else 0.0
flipped = r["flipped"] if (r and r["flipped"] is not None) else 0.0
check("delay_s matches actual_departure - aimed_departure", agree >= 0.85,
      f"agree {agree:.1%}, sign flipped {flipped:.1%}")

PASS | delay_s matches actual_departure - aimed_departure | agree 100.0%, sign flipped 0.5%


In [8]:
print("Building Dimension Tables...")

# Derive dim_line dynamically from clean_stop_visits
dim_line = sv.select("line_ref", "route_short_name", "route_type_name").dropDuplicates(["line_ref"])
emit("dim_line", dim_line)

# Derive dim_stop dynamically from clean_stop_visits
dim_stop = sv.select("stop_ref", "stop_name", "stop_lat", "stop_lon").dropDuplicates(["stop_ref"])
emit("dim_stop", dim_stop)

d = F.to_date("service_date")
emit("dim_date", spark.createDataFrame([(x,) for x in dates], "service_date string")
     .withColumns({"weekday": weekday_mon0(d), "weekday_name": F.date_format(d, "EEEE")})
     .withColumn("is_weekend", F.col("weekday") >= 5)
     .orderBy("service_date"))

emit("dim_hour_band", spark.createDataFrame(sorted(BAND_OF_HOUR.items()), "local_hour int, hour_band string"))
emit("gold_buckets", spark.createDataFrame(
    bucket_table("delay_s", "d", DELAY_EDGES) + bucket_table("time_lost_s", "tl", TL_EDGES),
    "metric string, bucket string, lo int, hi int, label string"))

Building Dimension Tables...


Py4JJavaError: An error occurred while calling o436.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [ ]:
print("Building Core Gold Aggregations & Cubes...")
HIST = hist_cols("d", "delay_s", "valid", DELAY_EDGES)
MEASURES = f"""
  COUNT(*) AS n_calls,
  SUM(CAST(is_actual AS INT)) AS n_actual,
  SUM(CAST((is_actual AND delay_outlier) AS INT)) AS n_outlier,
  SUM(CAST(valid AS INT)) AS n_valid,
  SUM(CAST((valid AND delay_s < {EARLY_S}) AS INT)) AS n_early,
  SUM(CAST((valid AND delay_s >= {EARLY_S} AND delay_s < {LATE_S}) AS INT)) AS n_on_time,
  SUM(CAST((valid AND delay_s >= {LATE_S}) AS INT)) AS n_late,
  COALESCE(SUM(CASE WHEN valid THEN delay_s END), 0) AS sum_delay_s,
  COALESCE(SUM(CASE WHEN valid THEN delay_s * delay_s END), 0) AS sum_delay_sq,
  {HIST}"""

cube_dims = "service_date, local_hour, hour_band, weekday, line_ref, direction_ref"
emit("gold_visits_cube", spark.sql(f"SELECT {cube_dims}, {MEASURES} FROM sv GROUP BY {cube_dims}"))

emit("gold_stop_cube", spark.sql(f"""
    SELECT service_date, hour_band, line_ref, stop_ref, {MEASURES} 
    FROM sv GROUP BY service_date, hour_band, line_ref, stop_ref"""))

emit("gold_stop_order_delay", spark.sql(f"""
    SELECT service_date, hour_band, line_ref, direction_ref, stop_order,
      COUNT(*) AS n_calls, SUM(CAST(valid AS INT)) AS n_valid,
      SUM(CAST((valid AND delay_s < {EARLY_S}) AS INT)) AS n_early,
      SUM(CAST((valid AND delay_s >= {LATE_S}) AS INT)) AS n_late,
      COALESCE(SUM(CASE WHEN valid THEN delay_s END), 0) AS sum_delay_s,
      COALESCE(SUM(CASE WHEN valid THEN delay_s * delay_s END), 0) AS sum_delay_sq
    FROM sv GROUP BY service_date, hour_band, line_ref, direction_ref, stop_order"""))

emit("fact_stop_visits", sv.select(
    "service_date", "local_hour", "hour_band", "weekday", "journey_ref", "line_ref", "direction_ref",
    "stop_order", "stop_ref", "aimed_departure", "actual_departure", "delay_s", "valid", "delay_outlier"), one_file=False)

In [ ]:
print("Building Segment Runtime & Headway Analytics...")
SECS = "(unix_micros({a}) - unix_micros({b})) / 1e6"

emit("gold_segment_runtime", spark.sql(f"""
    WITH v AS (SELECT * FROM sv WHERE valid),
    p AS (SELECT *, LAG(stop_ref) OVER w AS from_stop_ref, LAG(stop_order) OVER w AS prev_order,
                 LAG(aimed_departure) OVER w AS prev_aimed, LAG(actual_departure) OVER w AS prev_obs
          FROM v WINDOW w AS (PARTITION BY service_date, journey_ref ORDER BY stop_order)),
    s AS (SELECT *, {SECS.format(a='actual_departure', b='prev_obs')} AS act_s,
                    {SECS.format(a='aimed_departure', b='prev_aimed')} AS aim_s
          FROM p WHERE prev_order = stop_order - 1)
    SELECT service_date, hour_band, line_ref, from_stop_ref, stop_ref AS to_stop_ref,
           COUNT(*) AS n, SUM(act_s) AS sum_actual_s, SUM(aim_s) AS sum_aimed_s, SUM(act_s * act_s) AS sum_actual_sq
    FROM s WHERE act_s BETWEEN 0 AND {SEG_MAX_S} AND aim_s > 0
    GROUP BY service_date, hour_band, line_ref, from_stop_ref, stop_ref"""))

emit("gold_headway", spark.sql(f"""
    WITH v AS (SELECT * FROM sv WHERE valid),
    p AS (SELECT *, LAG(aimed_departure) OVER w AS pa, LAG(actual_departure) OVER w AS po
          FROM v WINDOW w AS (PARTITION BY service_date, line_ref, direction_ref, stop_ref ORDER BY aimed_departure)),
    h AS (SELECT service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref,
                 {SECS.format(a='aimed_departure', b='pa')} AS aim_h, {SECS.format(a='actual_departure', b='po')} AS act_h
          FROM p WHERE pa IS NOT NULL)
    SELECT service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref,
           COUNT(*) AS n_pairs, SUM(act_h) AS sum_actual_headway_s, SUM(aim_h) AS sum_aimed_headway_s,
           SUM(CAST((act_h < {BUNCH_RATIO} * aim_h) AS INT)) AS n_bunched,
           SUM(CAST((act_h > {GAP_RATIO} * aim_h) AS INT)) AS n_large_gap
    FROM h WHERE aim_h BETWEEN {HEADWAY_MIN_S} AND {HEADWAY_MAX_S}
    GROUP BY service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref"""))

In [ ]:
print("Processing Vehicle Activity and GPS Evidence Tables...")

# 1. Fleet activity from trip_vehicle & stop_gps_status
tv_raw = read(SILVER_DIR, "trip_vehicle", required=False)
if tv_raw is not None:
    emit("gold_fleet_activity", tv_raw.groupBy("service_date").agg(
        F.count("*").alias("n_trips"),
        F.countDistinct("vehicle_id").alias("n_vehicles"),
        F.sum("n_stops").alias("sum_stops"),
        F.avg("median_m").alias("avg_median_m")
    ).orderBy("service_date"))

# 2. Stop GPS status evidence from v6
gps_raw = read(SILVER_DIR, "stop_gps_status", required=False)
if gps_raw is not None:
    ev = gps_raw.withColumns({
        "visit_time": to_utc(gps_raw, "visit_time"),
        "line_ref": F.col("route_id").cast("string"),
        "stop_ref": F.col("stop_id").cast("string"),
        "service_date": F.col("service_date").cast("string"),
        "time_lost_s": num(gps_raw, "time_lost_s"),
        "p_excess": num(gps_raw, "p_excess"),
        "ts_offset_enter_s": num(gps_raw, "ts_offset_enter_s"),
        "ts_offset_leave_s": num(gps_raw, "ts_offset_leave_s"),
    })
    off_ok = (F.abs("ts_offset_enter_s") <= 300) & (F.abs("ts_offset_leave_s") <= 300)
    ev = ev.withColumns({
        "local_hour": F.coalesce(F.hour(local("visit_time")), F.lit(-1)),
        "off_enter": F.when(off_ok, F.col("ts_offset_enter_s")),
        "off_leave": F.when(off_ok, F.col("ts_offset_leave_s")),
    }).withColumn("hour_band", hour_band(F.col("local_hour"))).cache()
    ev.createOrReplaceTempView("ev")
    emit("gold_stop_evidence", spark.sql(f"""
        SELECT service_date, hour_band, line_ref, stop_ref, gps_class, COUNT(*) AS n_calls,
               COUNT(time_lost_s) AS n_time_lost, COALESCE(SUM(time_lost_s), 0) AS sum_time_lost_s,
               SUM(CAST((p_excess <= 0.05) AS INT)) AS n_p_excess_le_05,
               {hist_cols('tl', 'time_lost_s', 'time_lost_s IS NOT NULL', TL_EDGES)}
        FROM ev GROUP BY service_date, hour_band, line_ref, stop_ref, gps_class"""))

# 3. Time lost null reference
nul = read(SILVER_DIR, "time_lost_null", required=False)
if nul is not None and nul.limit(1).count():
    x = num(nul, "time_lost_s")
    bucket = sum(F.when(x >= e, 1).otherwise(0) for e in TL_EDGES)
    counts = dict(nul.where(x.isNotNull()).groupBy(bucket.alias("b")).count().collect())
    emit("gold_time_lost_reference", spark.createDataFrame(
        [(f"tl{i}", int(counts.get(i, 0))) for i in range(len(TL_EDGES) + 1)],
        "bucket string, n_stop_free_intervals long"))

In [ ]:
print("Running Automated Data Reconciliation Audit...")
n_buckets = len(DELAY_EDGES) + 1
n_sv_count = spark.sql("SELECT SUM(n_calls) FROM gold_visits_cube").first()[0]
check("visits cube holds every call", n_sv_count == n_sv, f"Cube: {n_sv_count:,} | Expected: {n_sv:,}")
check("stop cube holds every call", spark.sql("SELECT SUM(n_calls) FROM gold_stop_cube").first()[0] == n_sv)
check("early + on_time + late = valid", spark.sql("SELECT SUM(n_early + n_on_time + n_late - n_valid) FROM gold_visits_cube").first()[0] == 0)
check("histogram buckets add up to valid", spark.sql(f"SELECT SUM({' + '.join(f'd{i}' for i in range(n_buckets))} - n_valid) FROM gold_visits_cube").first()[0] == 0)
check("stop cube valid = visits cube valid", spark.sql("SELECT SUM(n_valid) FROM gold_stop_cube").first()[0] == spark.sql("SELECT SUM(n_valid) FROM gold_visits_cube").first()[0])

cube_sum = spark.sql("SELECT SUM(sum_delay_s) FROM gold_visits_cube").first()[0] or 0.0
row_sum = sv.where("valid").agg(F.sum("delay_s")).first()[0] or 0.0
check("sum of delays matches row-level table", abs(cube_sum - row_sum) <= 1e-4, f"Cube Sum: {cube_sum:,.1f}s | Row Sum: {row_sum:,.1f}s")

In [ ]:
print("Publishing Metadata & Summary Metrics...")
h = spark.sql("SELECT SUM(n_valid) v, SUM(n_early) e, SUM(n_on_time) o, SUM(n_late) l, SUM(sum_delay_s) s FROM gold_visits_cube").first()
if h and h.v:
    print(f"HEADLINE PERFORMANCE:")
    print(f"  Valid Calls: {int(h.v):,}")
    print(f"  Early: {h.e / h.v:.1%}")
    print(f"  On-Time: {h.o / h.v:.1%}")
    print(f"  Late: {h.l / h.v:.1%}")
    print(f"  Mean Delay: {h.s / h.v:.1f} s")

meta = {"built_at_utc": pd.Timestamp.now(tz="UTC").isoformat(), "engine": f"pyspark {spark.version}",
        "service_dates": ",".join(dates), "timezone": LOCAL_TZ,
        "checks_passed": str(sum(c["ok"] for c in CHECKS)), "checks_failed": str(sum(not c["ok"] for c in CHECKS))}
meta.update({f"rows_{k}": str(v) for k, v in TABLES.items()})
emit("gold_meta", spark.createDataFrame(list(meta.items()), "key string, value string"))

In [ ]:
# Optional DuckDB synchronization for dashboarding
try:
    import duckdb
except ImportError:
    duckdb = None
    print("DuckDB not installed: gold.duckdb skipped (Parquet tables are intact)")

if duckdb is not None:
    db_path = OUT / "gold.duckdb"
    db_path.unlink(missing_ok=True)
    con = duckdb.connect(str(db_path))
    for name in TABLES:
        p_path = (OUT / f"{name}.parquet").as_posix()
        if (OUT / name).is_dir():
            p_path = f"{p_path}/*.parquet"
        con.execute(f"CREATE TABLE {name} AS SELECT * FROM read_parquet('{p_path}')")
    con.close()
    print(f"gold.duckdb created successfully with {len(TABLES)} tables.")

In [ ]:
failed = [c for c in CHECKS if not c["ok"]]
if not failed:
    print(f"SUCCESS: ALL {len(CHECKS)} RECONCILIATION CHECKS PASSED. Gold layer ready at {OUT.resolve()}")
else:
    print(f"WARNING: {len(failed)} check(s) failed: {[c['check'] for c in failed]}")
assert not failed, f"Failed checks: {[c['check'] for c in failed]}"